# Curating systems for SLiM simulations

In [1]:
import json
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import shutil

while os.path.split(os.getcwd())[-1] != 'calvados-slim':
    os.chdir('..')

from src import curation, utils

/Users/frederikespersenknudsen/Software/Anaconda/anaconda3/envs/calvados/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Loading curations
with open('data/curation/systems.json') as file:
    systems = json.load(file)
systems_dir = os.path.abspath('data/curation/systems')

# Loading configurations
with open('data/configurations/conditions.json') as file:
    conditions = json.load(file)
with open('data/configurations/sampling.json') as file:
    sampling = json.load(file)
with open('data/configurations/go_scans.json') as file:
    go_scans = json.load(file)
slurm_scripts_dir = 'data/configurations/slurm'

In [4]:
# Curating input files for systems
failing = []
for system in tqdm(systems):
    system_dir = os.path.join(systems_dir, system['ID'])
    if os.path.exists(system_dir):
        continue

    assert system['Box sides'][0] < 100, "Failsafe for accidental box dimensions over 100 nm"

    # Getting missing CIF files
    if not os.path.exists(system['SLiMs'][0]['PDB']):
        path, extension = os.path.splitext(system['SLiMs'][0]['PDB'])
        pdb_id = os.path.basename(path)
        if extension == '.cif' and utils.is_pdb(pdb_id):
            utils.download_cif(pdb_id, system['SLiMs'][0]['PDB'])

    try:
        curation.System.curate(system, output_dir=systems_dir)
    except Exception as e:
        print(f"Curation failed for system '{system['ID']}': {e}")
        shutil.rmtree(os.path.join(system_dir))
        failing.append([system['ID'], e])

  5%|▌         | 15/285 [00:05<02:41,  1.67it/s]/Users/frederikespersenknudsen/Software/Anaconda/anaconda3/envs/calvados/lib/python3.10/site-packages/MDAnalysis/topology/guessers.py:146: UserWarning: Failed to guess the mass for the following atom types: SE
  warnings.warn("Failed to guess the mass for the following atom types: {}".format(atom_type))
  9%|▉         | 27/285 [00:14<03:18,  1.30it/s]

Curation failed for system 'traf2-tnfr2': 82


 10%|█         | 29/285 [00:15<03:11,  1.34it/s]

Curation failed for system 'wdr5-mll2': 0


 19%|█▉        | 55/285 [00:36<02:44,  1.40it/s]

Curation failed for system 'cbfa2t1-ncor2': The sequence extracted by CALVADOS from '/Users/frederikespersenknudsen/Repositories/calvados-slim/data/curation/systems/cbfa2t1-ncor2/mynd.pdb' does not match the construct specified for chain 'mynd'


... (25 outputs left out)

Curation failed for system 'nsp1-irf3': The sequence extracted by CALVADOS from '/Users/frederikespersenknudsen/Repositories/calvados-slim/data/curation/systems/nsp1-irf3/nsp1.pdb' does not match the construct specified for chain 'nsp1'


 99%|█████████▉| 282/285 [03:51<00:04,  1.53s/it]/Users/frederikespersenknudsen/Software/Anaconda/anaconda3/envs/calvados/lib/python3.10/site-packages/MDAnalysis/topology/guessers.py:146: UserWarning: Failed to guess the mass for the following atom types: SE
  warnings.warn("Failed to guess the mass for the following atom types: {}".format(atom_type))
100%|█████████▉| 284/285 [03:52<00:01,  1.07s/it]

Curation failed for system 'kpna2-npm2': The sequence extracted by CALVADOS from '/Users/frederikespersenknudsen/Repositories/calvados-slim/data/curation/systems/kpna2-npm2/arm.pdb' does not match the construct specified for chain 'arm'


100%|██████████| 285/285 [03:54<00:00,  1.22it/s]
